<div dir="rtl">

# حدس کلمه‌ی بعدی

این دفترچه‌ی ژوپیتر (Jupyter notebook) ایده‌ی درس اول را با پایتون (Python) نشان می‌دهد: مدل زبانی به هر ادامه‌ی ممکن یک احتمال می‌دهد. اول احتمال‌هایی را می‌بینید که مدل کوچکی به اسم «آیا اکسپنس» (aya-expanse:8b) روی لپ‌تاپ خودم به ادامه‌ی ۱۲ ضرب‌المثل و دو جمله‌ی نیمه‌تمام داد. بعد مدل خیلی کوچکی می‌سازید که فقط کلمه‌ها را می‌شمارد و با همین شمارش جمله می‌سازد. کدها جز خود پایتون چیزی نمی‌خواهند؛ سلول‌ها را به ترتیب اجرا کنید.

</div>

<div dir="rtl">

## ۱. مدل به هر ادامه‌ی ممکن یک احتمال می‌دهد

کد زیر داده‌ی آزمایش را از فایل `data.json` کنار همین دفترچه می‌خواند. اگر فایل آنجا نباشد، نسخه‌ی کوچکی را به کار می‌برد که داخل خود کد نوشته شده است. بعد برای یکی از نیمه‌ها، ده تکه‌ای را که مدل از همه محتمل‌تر می‌دانست، هر کدام با یک نوار نشان می‌دهد. طول هر نوار احتمال آن تکه است و نوار پر یعنی صددرصد.

کنار تکه‌هایی که بدون فاصله می‌آیند نوشته شده «بدون فاصله». اگر رقم‌های انگلیسی را ترجیح می‌دهید، در کد مقدار `PERSIAN_DIGITS` را `False` کنید.

</div>

In [1]:
import json
from pathlib import Path

# رقم‌ها فارسی چاپ شوند یا نه
PERSIAN_DIGITS = True

# نسخه‌ی کوچکی از داده، برای وقتی که فایل داده کنار این دفترچه نیست
INLINE = {
    "prompts": [
        {
            "prompt": "هر که بامش بیش،",
            "usual_ending": "برفش بیشتر",
            "continuation": " برفش بیشتر",
            "top10": [
                {"token": " بر", "p": 0.9991},
                {"token": " برد", "p": 0.0006},
                {"token": "بر", "p": 0.0001},
                {"token": " وبر", "p": 0.0001},
                {"token": " برج", "p": 0.0},
                {"token": " در", "p": 0.0},
                {"token": " آس", "p": 0.0},
                {"token": " برون", "p": 0.0},
                {"token": " خر", "p": 0.0},
                {"token": " براد", "p": 0.0},
            ],
        },
        {
            "prompt": "امروز صبح رفتم",
            "top10": [
                {"token": " سر", "p": 0.0597},
                {"token": " بازار", "p": 0.0407},
                {"token": " به", "p": 0.0385},
                {"token": " خرید", "p": 0.0269},
                {"token": " تو", "p": 0.0235},
                {"token": " دانشگاه", "p": 0.0232},
                {"token": " یک", "p": 0.0227},
                {"token": " بانک", "p": 0.0212},
                {"token": " یه", "p": 0.0208},
                {"token": " کتابخانه", "p": 0.0194},
            ],
        },
    ]
}


# ده تکه‌ی محتمل بعدی بعد از یک نیمه، به شکل جفت تکه و احتمال
def candidates(item):
    top = item.get("top10") or item.get("next_token_top10") or []
    return [(c["token"], c["p"]) for c in top]


# اول فایل داده، اگر باشد؛ بعد هر نیمه‌ای که در فایل نیست، از نسخه‌ی کوچک
data = {}
path = Path("data.json")
found = path.exists()
if found:
    raw = json.loads(path.read_text(encoding="utf-8"))
    data = {item["prompt"]: item for item in raw["prompts"]}
for item in INLINE["prompts"]:
    data.setdefault(item["prompt"], item)

# نشانه‌ی نامرئی راست‌به‌چپ
RLM = "\u200f"
# نام تکه‌هایی که دیده نمی‌شوند
INVISIBLE = {"": "فاصله", "\u200c": "نیم‌فاصله", "\u0654": "همزه‌ی کوچک"}


# متن فارسی را در خروجی چپ‌به‌راست به ترتیب درست نگه می‌دارد
def rtl(text):
    return RLM + text + RLM


# رقم‌ها و ممیز را فارسی می‌کند، اگر رقم فارسی خواسته باشید
def digits(text):
    if not PERSIAN_DIGITS:
        return text
    return text.translate(str.maketrans("0123456789.", "۰۱۲۳۴۵۶۷۸۹٫"))


# احتمال را مثل متن درس گرد می‌کند: از ۹۹٫۵٪ به بالا با یک رقم اعشار و بقیه بدون اعشار.
# احتمال مدل هیچ‌وقت دقیقاً صفر یا صددرصد نیست، پس به جای ۰٪ و ۱۰۰٪ کلمه می‌نویسد.
def percent(p):
    sign = "٪" if PERSIAN_DIGITS else "%"
    if p >= 0.995:
        one = f"{p * 100:.1f}"
        if one == "100.0":
            return ("بیش از " if PERSIAN_DIGITS else "> ") + digits("99") + sign
        return digits(one) + sign
    whole = round(p * 100)
    if whole < 1:
        return ("کمتر از " if PERSIAN_DIGITS else "< ") + digits("1") + sign
    return digits(str(whole)) + sign


# سهم یک شمارش از کل؛ این عدد دقیق است، پس سه از سه یعنی صددرصد
def share(n, total):
    return digits(str(round(100 * n / total))) + ("٪" if PERSIAN_DIGITS else "%")


# نوار متنی: هر مربع پر یعنی ۲٫۵ درصد
def bar(p, width=40):
    full = round(p * width)
    return "█" * full + "░" * (width - full)


# تکه را داخل «» نشان می‌دهد و اگر بدون فاصله بیاید، یادداشتی کنارش می‌گذارد
def label(piece):
    text = INVISIBLE.get(piece.strip(), piece.strip())
    shown = "«" + text + "»"
    if not piece.startswith(" ") and piece[:1].isalpha():
        shown += " (بدون فاصله)"
    return rtl(shown)


# نیمه را پیدا می‌کند، حتی اگر «،» آخرش جا افتاده باشد
def find(start):
    wanted = start.strip().rstrip("،")
    for prompt, item in data.items():
        if prompt.rstrip("،") == wanted:
            return item
    print(rtl("این نیمه در داده نیست: «" + start + "»"))
    return None


# نوار احتمال تکه‌های بعد از یک نیمه را چاپ می‌کند
def show(start, how_many=10):
    item = find(start)
    if item is None:
        return
    print(rtl("بعد از «" + item["prompt"] + "»:"))
    for piece, p in candidates(item)[:how_many]:
        print(bar(p), percent(p).rjust(10), label(piece))
    if "continuation" in item:
        print(rtl("مدل نوشت: «" + item["prompt"] + item["continuation"] + "»"))
    if "usual_ending" in item:
        print(rtl("ما می‌گوییم: «" + item["prompt"] + " " + item["usual_ending"] + "»"))


if found:
    print(rtl("داده از فایل خوانده شد. نیمه‌هایی که در آن هست:"))
else:
    print(rtl("فایل داده پیدا نشد و نسخه‌ی کوچک به کار رفت. نیمه‌ها:"))
for prompt in data:
    print(rtl("  • " + prompt))
print()

show("هر که بامش بیش")

‏داده از فایل خوانده شد. نیمه‌هایی که در آن هست:‏
‏  • هر که بامش بیش،‏
‏  • از تو حرکت،‏
‏  • کبوتر با کبوتر،‏
‏  • قطره قطره جمع گردد،‏
‏  • شنیدن کی بود‏
‏  • دست بالای دست‏
‏  • نابرده رنج‏
‏  • کار نیکو کردن‏
‏  • جوینده‏
‏  • یک دست‏
‏  • تا تنور گرم است،‏
‏  • آب که از سر گذشت،‏
‏  • امروز صبح رفتم‏
‏  • دیروز با دوستم‏

‏بعد از «هر که بامش بیش،»:‏
████████████████████████████████████████      ۹۹٫۹٪ ‏«بر»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«برد»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«بر» (بدون فاصله)‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«وبر»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«برج»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«در»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«آس»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«برون»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«خر»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«براد»‏
‏مدل نوشت: «هر که بامش بیش، برفش

<div dir="rtl">

## ۲. مدل گاهی مطمئن است و گاهی نه

بعد از «هر که بامش بیش،» مدل به «بر» احتمال ۹۹٫۹٪ داد، یعنی تقریباً مطمئن بود. اما بعد از «امروز صبح رفتم» مطمئن نبود: محتمل‌ترین تکه «سر» بود، آن هم فقط با ۶٪ احتمال. کد زیر پنج تکه‌ی اول هر دو را زیر هم چاپ می‌کند تا فرق نوارها را ببینید.

</div>

In [2]:
for start in ("هر که بامش بیش", "امروز صبح رفتم"):
    show(start, how_many=5)
    print()

sure = candidates(find("هر که بامش بیش"))[0][1]
unsure = candidates(find("امروز صبح رفتم"))[0][1]
print(rtl("محتمل‌ترین تکه در اولی " + percent(sure) + " و در دومی فقط " + percent(unsure) + " است."))

‏بعد از «هر که بامش بیش،»:‏
████████████████████████████████████████      ۹۹٫۹٪ ‏«بر»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«برد»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«بر» (بدون فاصله)‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«وبر»‏
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ کمتر از ۱٪ ‏«برج»‏
‏مدل نوشت: «هر که بامش بیش، برفش بیشتر»‏
‏ما می‌گوییم: «هر که بامش بیش، برفش بیشتر»‏

‏بعد از «امروز صبح رفتم»:‏
██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░         ۶٪ ‏«سر»‏
██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░         ۴٪ ‏«بازار»‏
██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░         ۴٪ ‏«به»‏
█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░         ۳٪ ‏«خرید»‏
█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░         ۲٪ ‏«تو»‏

‏محتمل‌ترین تکه در اولی ۹۹٫۹٪ و در دومی فقط ۶٪ است.‏


<div dir="rtl">

## ۳. با شمردن کلمه‌ها هم می‌شود کلمه‌ی بعدی را حدس زد

ساده‌ترین مدل زبانی فقط می‌شمارد که در یک متن بعد از هر کلمه چه کلمه‌ای آمده است. متن زیر فقط از ضرب‌المثل‌های درس و چند جمله‌ی روزمره ساخته شده. کد سهم هر کلمه‌ی بعدی را با همان نوارها نشان می‌دهد: «رفتم» در متن شش بار هست و دو بار بعدش «نانوایی» آمده، پس سهم «نانوایی» دو از شش است. بعد از «بامش» همیشه «بیش» آمده، ولی بعد از «رفتم» چند کلمه‌ی مختلف.

این برنامه فقط کلمه‌ی قبلی را می‌بیند. مدل واقعی کل متنی را که تا اینجا نوشته شده می‌بیند و از حجم عظیمی از متن یاد گرفته است.

</div>

In [3]:
from collections import Counter, defaultdict

# ضرب‌المثل‌های درس و چند جمله‌ی روزمره
text = """
هر که بامش بیش، برفش بیشتر
از تو حرکت، از خدا برکت
کبوتر با کبوتر، باز با باز
قطره قطره جمع گردد، وانگهی دریا شود
شنیدن کی بود مانند دیدن
دست بالای دست بسیار است
نابرده رنج گنج میسر نمی‌شود
کار نیکو کردن از پر کردن است
جوینده یابنده است
یک دست صدا ندارد
تا تنور گرم است، نان را بچسبان
آب که از سر گذشت، چه یک وجب چه صد وجب
امروز صبح رفتم نانوایی و نان گرم خریدم
امروز صبح رفتم سر کار
امروز صبح رفتم دانشگاه و کلاس نبود
امروز صبح رفتم نانوایی و صف طولانی بود
دیروز با دوستم رفتم بازار و میوه خریدم
دیروز با دوستم رفتم کتابخانه
دیروز با دوستم حرف زدم
"""

after = defaultdict(Counter)
for line in text.strip().splitlines():
    words = line.replace("،", " ").split()
    for word, next_word in zip(words, words[1:]):
        after[word][next_word] += 1


# سهم هر کلمه‌ای را که بعد از این کلمه آمده چاپ می‌کند
def show_counts(word):
    counts = after.get(word)
    if not counts:
        print(rtl("در این متن بعد از «" + word + "» کلمه‌ای نیامده است."))
        return
    total = sum(counts.values())
    print(rtl("«" + word + "» در این متن " + digits(str(total)) + " بار آمده است. کلمه‌ی بعد از آن:"))
    for next_word, n in counts.most_common():
        print(bar(n / total), share(n, total).rjust(10), rtl("«" + next_word + "»"))


show_counts("بامش")
print()
show_counts("رفتم")

‏«بامش» در این متن ۱ بار آمده است. کلمه‌ی بعد از آن:‏
████████████████████████████████████████       ۱۰۰٪ ‏«بیش»‏

‏«رفتم» در این متن ۶ بار آمده است. کلمه‌ی بعد از آن:‏
█████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░        ۳۳٪ ‏«نانوایی»‏
███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░        ۱۷٪ ‏«سر»‏
███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░        ۱۷٪ ‏«دانشگاه»‏
███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░        ۱۷٪ ‏«بازار»‏
███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░        ۱۷٪ ‏«کتابخانه»‏


<div dir="rtl">

## ۴. جمله کلمه به کلمه ساخته می‌شود

با همین شمارش‌ها می‌شود جمله ساخت: از یک کلمه شروع کنید، کلمه‌ی بعدی را انتخاب کنید، به جمله اضافه کنید و دوباره انتخاب کنید. کد زیر با هر کدام از دو روش سه جمله می‌سازد. روش اول همیشه پرتکرارترین کلمه را برمی‌دارد، همان کاری که «آیا اکسپنس» در آزمایش درس می‌کرد. روش دوم به نسبت شمارش‌ها قرعه می‌کشد. دستور `random.seed(0)` قرعه‌ها را ثابت نگه می‌دارد تا خروجی شما با خروجی همین صفحه یکی باشد.

</div>

In [4]:
import random


# پرتکرارترین کلمه‌ی بعدی
def most_common_next(word):
    return after[word].most_common(1)[0][0]


# قرعه به نسبت شمارش‌ها
def drawn_next(word):
    counts = after[word]
    return random.choices(list(counts), weights=list(counts.values()))[0]


# از یک کلمه شروع می‌کند و تا هشت کلمه جلو می‌رود
def make_sentence(first_word, choose, max_words=8):
    words = [first_word]
    while len(words) < max_words and after.get(words[-1]):
        words.append(choose(words[-1]))
    return " ".join(words)


start = "امروز"

print(rtl("همیشه پرتکرارترین کلمه:"))
for _ in range(3):
    print(rtl("  " + make_sentence(start, most_common_next)))
print()

random.seed(0)
print(rtl("با قرعه:"))
for _ in range(3):
    print(rtl("  " + make_sentence(start, drawn_next)))

‏همیشه پرتکرارترین کلمه:‏
‏  امروز صبح رفتم نانوایی و نان را بچسبان‏
‏  امروز صبح رفتم نانوایی و نان را بچسبان‏
‏  امروز صبح رفتم نانوایی و نان را بچسبان‏

‏با قرعه:‏
‏  امروز صبح رفتم سر گذشت چه یک وجب‏
‏  امروز صبح رفتم دانشگاه و صف طولانی بود‏
‏  امروز صبح رفتم کتابخانه‏


<div dir="rtl">

روش اول هر بار همان جمله را می‌سازد، چون در هر قدم همیشه همان یک کلمه را برمی‌دارد؛ روش دوم کلمه‌های کم‌تکرارتر را هم گاهی برمی‌دارد و برای همین ممکن است هر بار جمله‌ی دیگری بسازد.

</div>

<div dir="rtl">

## این سه تغییر را خودتان امتحان کنید

۱. در کد بخش ۱، به جای «هر که بامش بیش» نیمه‌ی دیگری از فهرست بنویسید و ببینید مدل چقدر مطمئن بود.

۲. در کد بخش ۳، سه جمله از زندگی خودتان به متن اضافه کنید و دوباره ببینید بعد از «رفتم» چه می‌آید.

۳. در کد بخش ۴، عدد داخل `random.seed(0)` را عوض کنید و ببینید قرعه چه جمله‌های تازه‌ای می‌سازد.

</div>